In [ ]:
import datetime as dt

from pyproj import CRS

In [ ]:
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import ERA5Dataset, ERA5Var
from etdataset.icos import get_stations_config
from etdataset.utils import work_area_from_coord_point
from etdataset.validation_temp.temperature_rescaling import (
    filter_dataset_by_datetimes,
    generate_datetime,
    get_era5_grid,
    get_ta_td_celsius_at_location,
    prepare_temperature_inputs,
    read_era5_file,
    read_era5_files,
    temperature_rescaling_constant_lapse_rate,
)

In [ ]:
list_date = generate_datetime(dt.date(2023, 3, 3), dt.date(2023, 3, 4))

In [ ]:
list_date

In [ ]:
era5_xrds = read_era5_file(
    dt.datetime(2023, 3, 3, 0, 0), ERA5Dataset.ERA5, "out"
)

In [ ]:
era5_xrds = read_era5_files(list_date, "out")

In [ ]:
era5_xrds

In [ ]:
era5_filtered = filter_dataset_by_datetimes(era5_xrds, list_date)

In [ ]:
era5_filtered

In [ ]:
cfg = get_stations_config("DE-Geb")

In [ ]:
cfg

In [ ]:
bounds, crs_bounds = work_area_from_coord_point(
    cfg.lat, cfg.lon, 50000, 50000, CRS.from_epsg(4326)
)["lat/lon"]


In [ ]:
grid = get_era5_grid(bounds, crs_bounds, 0.25)

In [ ]:
dem, era5_dem, updated_data = prepare_temperature_inputs(
    data=grid,
    era5_data=era5_filtered,
    dataset=ERA5Dataset.ERA5,
)
print(dem, era5_dem)

In [ ]:
updated = temperature_rescaling_constant_lapse_rate(
    updated_data,
    dem,
    era5_filtered,
    era5_dem,
    variables=[
        ERA5Var.TEMPERATURE,
        ERA5Var.DEWPOINT_TEMPERATURE,
    ],
)

In [ ]:
updated

In [ ]:
ta, td = get_ta_td_celsius_at_location(updated, cfg.lon, cfg.lat)

In [ ]:
ta

In [ ]:
td

In [ ]:
mnt_path = "COP-DEM_GLO-30-DGED_S2_tiles/"


In [ ]:
roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
    cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
)["utm"]


In [ ]:
dem = get_dem_from_roi(
    roi_bbox=roi_bbox_utm,
    roi_crs=roi_crs_utm,
    base_dir=mnt_path,
    resolution=60,
)

In [ ]:
dem

In [ ]:
dem, era5_dem, updated_data = prepare_temperature_inputs(
    data=dem,
    era5_data=era5_filtered,
    dataset=ERA5Dataset.ERA5,
)


In [ ]:
updated = temperature_rescaling_constant_lapse_rate(
    updated_data,
    dem,
    era5_filtered,
    era5_dem,
    variables=[
        ERA5Var.TEMPERATURE,
        ERA5Var.DEWPOINT_TEMPERATURE,
    ],
)


In [ ]:
updated

In [ ]:
x, y = (
    work_area_from_coord_point(cfg.lat, cfg.lon, 0, 0, CRS.from_epsg(4326))[
        "utm"
    ][0].left,
    work_area_from_coord_point(cfg.lat, cfg.lon, 0, 0, CRS.from_epsg(4326))[
        "utm"
    ][0].bottom,
)


In [ ]:
ta, td = get_ta_td_celsius_at_location(updated, x, y)


In [ ]:
ta

In [ ]:
td